<div style='text-align: center;'>
<img src="images/math60082-banner.png" alt="image" width="80%" height="auto">
</div>

# Lab Workbook - Week 2

# Tasks

1. Integrate $f(x)=\sin(x)$ and $f(x)=\cos(x)$ over the region $\left[0,\frac{3}{4}\pi\right]$. Verify the accuracy of your results.


2. Consider that you are required to integrate the function $f(x)=\max(x,e^\frac{x}{2}-1)$ over the region $[0,5]$. How might you best deal with this problem?

Here we want to find the value $\alpha$ such that
$$
I = \int_0^5 \max(x,e^{x/2}-1) dx = \int_0^\alpha x dx + \int_\alpha^5 e^{x/2} - 1 dx
$$

Solve problem to find the root of $f(x) = x - (e^{x/2} - 1) $, at this point $x=e^{x/2}-1$. 

In [1]:
from scipy.optimize import root_scalar
from math import exp

In [2]:
fx = lambda x: x - exp(x/2) + 1
fx(1.0)

0.3512787292998718

In [3]:
fx(5)

-6.182493960703473

In [8]:
result = root_scalar( fx , x0=2 , x1=5.0 )

In [9]:
print(result)

      converged: True
           flag: converged
 function_calls: 9
     iterations: 8
           root: 2.5128624172523284


In [12]:
from scipy.integrate import quad 
alpha = result.root
I = quad( lambda x: max(x, exp(x/2.) - 1) , 0.0, 5.0)
print(I)
I1 = quad( lambda x: x , 0.0, alpha)
I2 = quad( lambda x: exp(x/2.) - 1,alpha,5.0)
print(I1,I2)

(18.009364282752735, 6.880362330199794e-08)
(3.1572387640196076, 3.5052391700536395e-14) (14.852125504154639, 1.6489171699334992e-13)


3. Experiment with the lower and upper limits to see what effect they have. Can you propose what would be the _best_ values to choose in this case? Explain your reasoning.

In [31]:
# function to integrate normal distribution multiplied by 1/(1+t^2)
def Gx_integrate( x ):
    lower_limit=-15
    upper_limit=15
    if x<lower_limit:
        return 0.
    elif x>upper_limit:
        return Gx_integrate(upper_limit)
    from scipy.integrate import quad as QUAD
    from math import exp,pi,sqrt
    return (1./sqrt(2.*pi))*QUAD(lambda t: exp(-t*t/2.)/(1+t*t) , lower_limit, x)[0]


In [32]:
print(f" {Gx_integrate(-3):12.8g} {Gx_integrate(0):12.8g} {Gx_integrate(3):12.8g}"  )

 0.00011622157   0.32783977   0.65556332


4. Try to write a function `Gxg_integrate( float: x , g )` which takes the `g` as an argument.

In [33]:
# function to integrate normal distribution multiplied by 1/(1+t^2)
def Gxg_integrate( x: float , g ):
    lower_limit=-15
    upper_limit=15
    if x<lower_limit:
        return 0.
    elif x>upper_limit:
        return Gxg_integrate(upper_limit,g)
    from scipy.integrate import quad as QUAD
    from math import exp,pi,sqrt
    return (1./sqrt(2.*pi))*QUAD(lambda t: exp(-t*t/2.) * g(x) , lower_limit, x)[0]


5. Test the efficiency of calculation for $N(x)$ against the version from the special function module in `scipy`.

In [34]:
from timeit import timeit
from scipy.special import ndtr as ND 
# function to integrate cumulative normal distribution
def Nx_integrate( x ):
    if x<-15.0:
        return 0.
    elif x>15.0:
        return 1.0
    from scipy.integrate import quad as QUAD
    from math import exp,pi,sqrt
    return 0.5 + (1./sqrt(2.*pi))*QUAD(lambda t: exp(-t*t/2.), 0, x)[0]

In [36]:
n = 100000
script="Gx_integrate(1.)"
timeIntegrate = timeit( script,number=n,globals=globals() )

print("Time taken to run ",n," calls to the function ",script, " is ", timeIntegrate," seconds.")

Time taken to run  100000  calls to the function  Gx_integrate(1.)  is  4.662803700004588  seconds.


In [37]:
def payoff(x):
    return 1/(1+x*x)

In [38]:
n = 100000
script="Gxg_integrate(1.,payoff)"
timeIntegrate = timeit( script,number=n,globals=globals() )

print("Time taken to run ",n," calls to the function ",script, " is ", timeIntegrate," seconds.")

Time taken to run  100000  calls to the function  Gxg_integrate(1.,payoff)  is  3.5872957999963546  seconds.


In [ ]:
n = 100000
script="ND(1.)"
timeIntegrate = timeit( script,number=n,globals=globals() )

print("Time taken to run ",n," calls to the function ",script, " is ", timeIntegrate," seconds.")

6. Test the efficiency of calculating $G(x)$ when $g$ is written inside the function, versus when $g$ is passed in as an argument. Does the flexibility of the second method come at a cost?